# 02 · GRPO with a controllable efficiency pressure λ
Reward = 1[correct] + λ·(1 − normalised length among the *correct* samples of the group); wrong or truncated = 0.
Dr.GRPO-style mean-centred advantages (λ is in reward units), exactly on-policy, LoRA r=32. **Staged with a gate** so a dead hypothesis costs little.
Stage lists (`stage1`, `stage2`, `stage_size`) come from the profile.

In [ ]:
import sys, json, os, glob, time; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lab import profiles, local, envs, collective, cost
P = profiles.get(); MAIN = P["main"]; HF = {t: v[0] for t, v in P["models"].items()}; FINAL = P["steps"]
SPEC = f"spec_{P['name']}.json"
print("PROFILE:", P["name"], "| GPU:", P["gpu"], "| cap $", P["cap"], "| main model:", HF[MAIN], "| steps:", FINAL)

In [ ]:
spec = json.load(open(SPEC)); train_spec = {f: spec[f] for f in envs.TRAIN_FAMILIES}
sps = json.load(open(f"sec_per_step_{P['name']}.json")); print(sps, train_spec)
def mk(tag, lam, seed, **kw):
    hf, mb = P["models"][tag]
    return dict(name=f"{tag}_lam{lam}_s{seed}", model=hf, lam=lam, seed=seed, steps=P["steps"], save_every=P["save_every"], n_prompts=P["n_prompts"],
                group=P["group"], lr=P["lr"], micro_bs=mb, spec=train_spec, **kw)
def logdf(name):
    fs = local.find(f"{name}.jsonl", must="logs"); return pd.read_json(fs[0], lines=True) if fs else None

## Stage 1 — the poles

In [ ]:
for tag, lam, seed in P["stage1"]: local.launch("train_run", mk(tag, lam, seed))
print(local.status(MAIN))

In [ ]:
local.pull("/logs"); print(local.status(MAIN))
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for tag, lam, seed in P["stage1"]:
    d = logdf(f"{tag}_lam{lam}_s{seed}")
    if d is None: continue
    for a, k in zip(ax, ["acc", "len", "trunc"]): a.plot(d.step, d[k].rolling(10, min_periods=1).mean(), label=f"λ={lam} s{seed}"); a.set_title(k)
ax[0].legend(); plt.show()

In [ ]:
def tail(name, k=None):
    d = logdf(name); k = k or max(5, len(d) // 4); return d.tail(k)[["acc", "len", "len_correct", "trunc"]].mean()
lo, hi = P["stage1"][0], P["stage1"][-1]
a, b = tail(f"{lo[0]}_lam{lo[1]}_s{lo[2]}"), tail(f"{hi[0]}_lam{hi[1]}_s{hi[2]}"); print(pd.DataFrame({"lam_lo": a, "lam_hi": b}).round(3))
g = P["gate"]; GATE = bool(b.len < g["len_ratio"] * a.len and b.acc > a.acc - g["acc_drop"])
print(f"GATE 1 (length ×<{g['len_ratio']} and accuracy drop <{g['acc_drop']}):", GATE)

**If GATE 1 fails:** raise `lr` or the top λ in `lab/profiles.py`, redeploy, rerun Stage 1 only. Do not continue.

In [ ]:
if GATE:
    for tag, lam, seed in P["stage2"]: local.launch("train_run", mk(tag, lam, seed))
print(local.status(MAIN))

**Pilot noise floor.** In the pilot, `stage2` contains a second λ=0 seed. The gap between the two λ=0 runs below is your seed noise — if it is as large as the λ effect,
the paper needs more seeds/steps, not more λ values.

In [ ]:
local.wait(MAIN, every=60); local.pull("/logs")
runs0 = [r for r in local.status(MAIN)]; rows = {r: tail(r) for r in runs0 if logdf(r) is not None}
print(pd.DataFrame(rows).T.round(3).sort_index())

## Size axis (paper profile only: other model sizes)

In [ ]:
if GATE and P["stage_size"]:
    for tag, lam, seed in P["stage_size"]: local.launch("train_run", mk(tag, lam, seed))
    print(local.wait("m", every=120)); local.pull("/logs")
df, spent = local.ledger(); print(df.groupby("fn").usd.sum().round(2), f"\nSPENT: ${spent:.2f} of ${P['cap']:.0f}")